#### Decodificando la Ley: Clasificación Inteligente y Búsqueda Semántica de Jurisprudencia Argentina

##### TP1 — Análisis y visualización de datos

Importamos las librerías y definimos la configuración base.

El dataset completo se descarga con el notebook `descarga-lectura-dataset.ipynb` de la raíz del repositorio.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

# Rutas relativas al notebook: os.path.join arma el separador correcto en Windows y Unix
DATASET_PATH = os.path.join("..", "datos", "full-dataset")
DATASET_FILE = os.path.join(DATASET_PATH, "dataset.jsonl")

# Semilla fija: cualquier muestreo aleatorio da el mismo resultado en cada corrida
RANDOM_SEED = 42

# Un solo color para todos los gráficos: cada marca es la misma magnitud medida sobre
# categorías distintas, no series distintas. Colorear por categoría no agregaría información.
COLOR_DATO = "#4C78A8"

# max_columns=None desactiva el "..." que pandas mete cuando hay muchas columnas
pd.set_option("display.max_columns", None)
# max_colwidth limita el ancho de cada celda: los fallos son textos largos y sin esto la tabla es ilegible
pd.set_option("display.max_colwidth", 120)

##### 1. Lectura del dataset completo

El archivo `dataset.jsonl` pesa ~2.5 GB en disco. `pd.read_json` construye el `DataFrame` entero en
memoria con copias intermedias, así que el pico de RAM es varias veces el tamaño final: en un equipo
con poca memoria libre el sistema empieza a paginar a disco y la celda pasa de tardar minutos a
tardar media hora.

Las secciones 2 a 4 necesitan las 71 columnas — analizar los nulos de todas las variables es
justamente su objetivo. La **sección 5 no**: usa 4 columnas y se carga por su cuenta en ~1 minuto,
así que se puede correr sin ejecutar esta celda.

Si el equipo no tiene RAM suficiente para las secciones 2 a 4, usar `datos/dataset_sample.jsonl.gz`
(muestra del 1%) o leer por chunks con `pd.read_json(..., lines=True, chunksize=...)`.

In [ ]:
# lines=True lee JSON Lines: un objeto JSON por línea, no un único array
# El archivo pesa ~2.5 GB, así que esta celda tarda varios minutos y consume bastante RAM
df = pd.read_json(DATASET_FILE, lines=True)

# shape devuelve (filas, columnas); el formato :, agrega separador de miles
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

##### 2. 10 filas aleatorias

Fijamos `random_state` para que la muestra sea reproducible entre ejecuciones.

In [ ]:
# random_state fija la semilla del muestreo para que estas 10 filas sean siempre las mismas
df.sample(10, random_state=RANDOM_SEED)

,numero-sumario,materia,sumario,descriptores,referencias-normativas,texto,fuente,analista,responsable,fecha,tipo-tribunal,instancia,jurisdiccion,provincia,caratula,fecha-alta,fecha-mod,uid-alta,uid-mod,timestamp,timestamp-m,timestamp-alta,id-infojus,fecha-umod,titulo,guid,numero-fallo,tribunal,pais,tipo-fallo,localidad,magistrados,actor,demandado,sobre,sumarios-relacionados,texto-doc,sala,numero-interno,tribunal-origen,publicacion,texto-completo,numero-camara,citas,jurisprudencia-vinculada,sintesis,hechos,identificacion-plenario,doctrina-relacionada,referencias-juris,control-constitucional,disidencia,esq-clasificacion,seq,archivo,tipo-norma,org_emisor,fecha_alta,ppublic_nov,d_link,fecha_umod,status,titulo_noticia,destacada,fecha_newsletter,suplemento_bo,subtipo,url_portal,nro_orden,fecha_destacada,sigla_emisor
613210,NaN,NaN,NaN,"{'descriptor': [{'elegido': {'termino': 'Término elegido para describir al caso'}, 'preferido': {'termino': 'Término...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,123456789-0abc-defg9415-000ssoiramus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
195840,C0005865,CIVIL,[[p]]LOCACION DE INMUEBLES-CONVENIO DE DESOCUPACION:REQUISITOS[[/p]],"{'descriptor': [{'elegido': {'termino': 'locación de inmuebles'}, 'preferido': {'termino': 'Obligaciones y contratos...","{'referencia-normativa': [{'cr': None, 'id': 'REFERENCIAS_NORMATIVAS_0', 'ref': 'LEY C 021342 0000 00 00 0047 000 00...",[[p]]Si bien el art. 47 de la ley 21.342 establece que la homologación de un convenio de desocupación se dictar...,OFICIAL,NaN,NaN,1990-03-26,CI,C,"{'codigo': 'NACIONAL', 'descripcion': 'Nacional', 'capital': 'Nacional', 'id-pais': 11}",Ciudad Autónoma de Buenos Aires,"OLIVIERI, N. c/ BENITEZ, H. s/ HOMOLOGACION",1995-05-02,2013-05-20,SAUID,ABM,2013-05-20 15:26:51,2013-05-20 15:26:51,NaT,SUC0005865,NaN,"Locación de inmuebles, convenio de desocupación",123456789-0abc-defg5685-000csoiramus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,F,R000061718,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98445,B0035373,CIVIL - COMERCIAL,[[p]]RECURSO EXTRAORDINARIO;SENTENCIA RECURRIBLE:ACTUALIZACION MONETARIA[[/p]],"{'descriptor': [{'elegido': {'termino': 'recurso extraordinario'}, 'preferido': {'termino': 'Derecho procesal/recurs...","{'referencia-normativa': {'cr': None, 'id': 'REFERENCIAS_NORMATIVAS_0', 'ref': 'CPC B 007425 1968 09 19 0278 000 027...","[[p]]Es recurrible por vía extraordinaria (art. 278, CPC) la resolución recaída en ejecución de sentencia, cu...",OFICIAL,NaN,NaN,1992-04-28|1991-03-19|1991-08-20|1993-03-30,CS CS CS CS,S S S,"{'codigo': 'LOCAL', 'descripcion': 'Local', 'capital': 'Local', 'id-pais': 11}",Buenos Aires,"Campana, Francisco D. c/ De Benedittis, Eduardo s/ Daños y perjuicios. Recurso de queja Finansur SACF c/ Bohoslavsky...",1996-12-13,2014-06-05,SAUID,SASAIJ,2014-06-05 19:48:10,2014-06-05 19:48:10,NaT,SUB0035373,NaN,"Recurso extraordinario, resoluciones recurribles, actualización monetaria",123456789-0abc-defg3735-300bsoiramus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ac49842 Ac46663 Ac48638 Ac52501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
807045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Belloro, Mariana",2021-06-09,CS,S,"{'codigo': 'LOCAL', 'descripcion': 'Local', 'capital': 'Local', 'id-pais': 11}",Ciudad Autónoma de Buenos Aires,NaN,2022-01-11,2022-01-11,ABM,ABM,2022-01-11 15:09:14,2022-01-11 15:09:14,2022-01-11 14:19:06,FA21380107,NaN,NaN,123456789-701-0831-2ots-eupmocsollaf,21380107.0,TRIBUNAL SUPERIOR DE JUSTICIA DE LA CIUDAD DE BUENOS AIRES,ARGENTINA,SENTENCIA,CIUDAD DE BUENOS AIRES,Weinberg-Ruiz-De Langhe-Otamendi-Lozano,"Incidente de incompetencia en autos NN, NN sobre 153

##### 3. Listado de columnas

Además del nombre, miramos el tipo inferido por pandas: es el primer indicio de qué variables
son categóricas, cuáles son texto libre y cuáles son estructuras anidadas (listas o diccionarios).

In [ ]:
# start=1 numera desde 1 en lugar de 0, solo para que el listado se lea más cómodo
for i, columna in enumerate(df.columns, start=1):
    # :>2 alinea el número a la derecha en 2 caracteres; dtype es el tipo que pandas infirió
    print(f"{i:>2}. {columna} ({df[columna].dtype})")

 1. numero-sumario (object)
 2. materia (str)
 3. sumario (str)
 4. descriptores (object)
 5. referencias-normativas (object)
 6. texto (str)
 7. fuente (str)
 8. analista (str)
 9. responsable (str)
10. fecha (object)
11. tipo-tribunal (str)
12. instancia (str)
13. jurisdiccion (object)
14. provincia (str)
15. caratula (str)
16. fecha-alta (str)
17. fecha-mod (str)
18. uid-alta (str)
19. uid-mod (str)
20. timestamp (datetime64[us])
21. timestamp-m (datetime64[us])
22. timestamp-alta (datetime64[us])
23. id-infojus (str)
24. fecha-umod (float64)
25. titulo (str)
26. guid (str)
27. numero-fallo (float64)
28. tribunal (str)
29. pais (str)
30. tipo-fallo (str)
31. localidad (str)
32. magistrados (object)
33. actor (object)
34. demandado (object)
35. sobre (object)
36. sumarios-relacionados (object)
37. texto-doc (object)
38. sala (object)
39. numero-interno (object)
40. tribunal-origen (object)
41. publicacion (object)
42. texto-completo (object)
43. numero-camara (object)
44. citas (obje

In [ ]:
# verbose=True fuerza el listado completo de columnas (pandas lo colapsa cuando son muchas)
# show_counts=True garantiza la columna "Non-Null Count", que se omite en DataFrames grandes
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 874845 entries, 0 to 874844
Data columns (total 71 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   numero-sumario            483305 non-null  object        
 1   materia                   483305 non-null  str           
 2   sumario                   483305 non-null  str           
 3   descriptores              589273 non-null  object        
 4   referencias-normativas    237528 non-null  object        
 5   texto                     483295 non-null  str           
 6   fuente                    485286 non-null  str           
 7   analista                  112108 non-null  str           
 8   responsable               162212 non-null  str           
 9   fecha                     769745 non-null  object        
 10  tipo-tribunal             767118 non-null  str           
 11  instancia                 766558 non-null  str           
 12  jurisdiccion 

##### 4. Nulos por variable

`isna()` cuenta `None` / `NaN`, pero **no** cuenta strings vacíos ni listas vacías, que en este dataset
también representan ausencia de dato. Por eso medimos ambas cosas: nulos reales y "vacíos".

In [ ]:
# Total de registros: lo usamos como denominador para los porcentajes
total_filas = len(df)

# Armamos una tabla resumen donde cada fila es una variable del dataset
nulos = pd.DataFrame(
    {
        # dtype como string para que entre prolijo en la tabla
        "dtype": df.dtypes.astype(str),
        # isna() marca True en los faltantes; sum() los cuenta por columna
        "nulos": df.isna().sum(),
        # notna() es el complemento: cuántos valores presentes tiene cada variable
        "no_nulos": df.notna().sum(),
    }
)
# Porcentaje de faltantes: más comparable entre variables que el conteo crudo
nulos["pct_nulos"] = (nulos["nulos"] / total_filas * 100).round(2)

# option_context levanta el límite de filas solo para este display: evita truncar las variables
# sin dejar el estado global del notebook modificado para el resto de las celdas
with pd.option_context("display.max_rows", None):
    # Ordenamos de mayor a menor para ver primero las variables más incompletas
    display(nulos.sort_values("nulos", ascending=False))

,dtype,nulos,no_nulos,pct_nulos
control-constitucional,str,874844,1,100.00
referencias-juris,object,874844,1,100.00
disidencia,str,874844,1,100.00
doctrina-relacionada,str,874844,1,100.00
esq-clasificacion,str,874844,1,100.00
sigla_emisor,str,874843,2,100.00
url_portal,str,874839,6,100.00
fecha_destacada,str,874835,10,100.00
nro_orden,float64,874835,10,100.00
fecha_newsletter,str,874700,145,99.98


In [ ]:
def es_vacio(valor):
    """True si el valor es nulo, un string en blanco o una colección vacía."""
    # Listas y diccionarios vacíos: pd.isna() no los detecta, hay que medir su longitud
    if isinstance(valor, (list, tuple, dict, set)):
        return len(valor) == 0
    # Strings con solo espacios cuentan como ausencia de dato
    if isinstance(valor, str):
        return valor.strip() == ""
    # Para el resto alcanza con el chequeo estándar de pandas
    return pd.isna(valor)


# map() aplica la función celda por celda: es la operación más costosa del notebook.
# Guardamos la máscara booleana completa para reutilizarla y no recalcularla más abajo.
mask_vacios = df.map(es_vacio)

# sum() sobre la máscara cuenta los True por columna
vacios = mask_vacios.sum()

# Comparamos ambas mediciones: la diferencia son los "vacíos" que isna() no ve
faltantes = pd.DataFrame(
    {
        "nulos": nulos["nulos"],
        "nulos_o_vacios": vacios,
    }
)
faltantes["pct_nulos_o_vacios"] = (faltantes["nulos_o_vacios"] / total_filas * 100).round(2)

# Misma tabla por variable: mostramos todas las filas sin truncar
with pd.option_context("display.max_rows", None):
    display(faltantes.sort_values("nulos_o_vacios", ascending=False))

,nulos,nulos_o_vacios,pct_nulos_o_vacios
control-constitucional,874844,874844,100.00
referencias-juris,874844,874844,100.00
disidencia,874844,874844,100.00
doctrina-relacionada,874844,874844,100.00
esq-clasificacion,874844,874844,100.00
sigla_emisor,874843,874843,100.00
url_portal,874839,874839,100.00
fecha_destacada,874835,874835,100.00
nro_orden,874835,874835,100.00
fecha_newsletter,874700,874700,99.98


Y una mirada por fila: cuántas variables le faltan a cada registro.

In [ ]:
# axis=1 suma a lo ancho: cuántos campos le faltan a CADA registro.
# Reutilizamos mask_vacios (misma vara que la celda anterior): cuenta nulos, strings
# en blanco y colecciones vacías. Con df.isna() un campo "" o [] contaría como presente.
faltantes_por_fila = mask_vacios.sum(axis=1)

# Registros completos: los más útiles para modelar sin imputar nada
print(f"Filas sin ningún faltante: {(faltantes_por_fila == 0).sum():,}")
print(f"Total de columnas: {df.shape[1]}")
print("\nDistribución de faltantes por fila:")

# La distribución puede tener tantas categorías como columnas, así que la mostramos completa
with pd.option_context("display.max_rows", None):
    # value_counts cuenta cuántas filas tienen N faltantes; sort_index las ordena por N ascendente
    display(faltantes_por_fila.value_counts().sort_index())

Filas sin ningún faltante: 0
Total de columnas: 71

Distribución de faltantes por fila:


39         3
40        64
41       523
42      1215
43      8339
44     22237
45     60813
46     98880
47    196408
48    226620
49    123135
50     25872
51      5426
52       119
53        31
54        49
55        13
56         1
59         1
69    105096
Name: count, dtype: int64

##### 5. Desbalance de clases

Miramos las cuatro variables candidatas a etiqueta del clasificador: `materia`, `descriptores`,
`tipo-tribunal` y `jurisdiccion`. Cada una tiene una forma distinta en el JSON y ninguna se puede
contar con un `value_counts()` directo:

| variable | forma en el dataset | tipo de problema |
|---|---|---|
| `materia` | string, a veces con varias materias unidas por guion | multiclase (una etiqueta por sumario) |
| `descriptores` | dict anidado con una lista de términos | multietiqueta (varios términos por sumario) |
| `tipo-tribunal` | string con código corto (`CS`, `CC`, `LB`, ...) | multiclase |
| `jurisdiccion` | dict con `codigo`, `descripcion`, `capital`, `id-pais` | multiclase |

De cada una nos interesan tres números, más informativos que el gráfico: el **ratio entre la clase
mayoritaria y la minoritaria**, **cuántas clases tienen menos de 50 ejemplos** (no se pueden aprender
ni validar con partición estratificada) y la **cobertura acumulada** del top-N.

Esta sección carga su propio `DataFrame` con las 4 columnas que necesita, en lugar de reusar el `df`
de 71 columnas de la sección 1. No es una optimización prematura: son ~1 minuto y 0,3 GB de RAM
contra los varios GB del `df` completo, y permite trabajar la sección de forma independiente.

In [ ]:
# Las únicas columnas que necesita esta sección
COLUMNAS_CLASES = ["materia", "descriptores", "tipo-tribunal", "jurisdiccion"]


def leer_columnas(ruta, columnas):
    """Lee un JSON Lines quedándose solo con `columnas`.

    `pd.read_json` no tiene `usecols`: carga el archivo entero sí o sí. Parseando línea a
    línea con `json.loads` descartamos el resto de los campos ANTES de construir el
    DataFrame, que es donde está el ahorro de memoria.
    """
    registros = []
    with open(ruta, encoding="utf-8") as f:
        for linea in f:
            registro = json.loads(linea)
            # get() con default None: no todos los registros traen las 4 claves
            registros.append({c: registro.get(c) for c in columnas})
    return pd.DataFrame(registros)


df_clases = leer_columnas(DATASET_FILE, COLUMNAS_CLASES)

# Cantidad de registros: mismo universo que el df completo, sirve de control
total_registros = len(df_clases)
print(f"Filas: {total_registros:,}")
print(f"Columnas: {df_clases.shape[1]}")
# deep=True mide el tamaño real de los objetos de Python, no solo el de los punteros
print(f"RAM: {df_clases.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# Umbral por debajo del cual una clase es inviable: con menos ejemplos no alcanza
# para partir en train/test estratificado y que queden casos en ambos lados.
MIN_EJEMPLOS_CLASE = 50


def resumen_desbalance(clases, nombre, top=25):
    """Tabla de frecuencias + métricas de desbalance para una serie ya desanidada.

    `clases` es una Series donde cada fila es UNA etiqueta. En variables multietiqueta
    hay más filas que registros, por eso informamos ambos totales.
    """
    conteo = clases.value_counts()
    total_etiquetas = len(clases)

    # Porcentaje sin redondear: es la base de todos los cálculos
    pct = conteo / total_etiquetas * 100

    resumen = pd.DataFrame({"n": conteo})
    # Porcentaje sobre el total de etiquetas: comparable entre variables de distinto tamaño
    resumen["pct"] = pct.round(2)
    # El acumulado se calcula ANTES de redondear. Con miles de clases, sumar porcentajes ya
    # redondeados a 2 decimales pierde tanto en la cola que el acumulado nunca llega a 100.
    resumen["pct_acum"] = pct.cumsum().round(2)

    raras = (conteo < MIN_EJEMPLOS_CLASE).sum()

    print(f"--- {nombre} ---")
    print(f"Etiquetas totales: {total_etiquetas:,}")
    print(f"Clases distintas:  {len(conteo):,}")
    print(f"Clase mayoritaria: {conteo.index[0]} ({conteo.iloc[0]:,} | {resumen['pct'].iloc[0]}%)")
    # max/min es la medida directa de cuán torcida está la distribución.
    # Con colas de clases únicas el mínimo es 1 y el ratio equivale al conteo de la mayoritaria.
    print(f"Ratio mayoritaria/minoritaria: {conteo.max() / conteo.min():,.0f}x")
    print(f"Clases con menos de {MIN_EJEMPLOS_CLASE} ejemplos: {raras:,} ({raras / len(conteo) * 100:.1f}% de las clases)")

    # min() recorta cada N a la cantidad real de clases; sorted + dict.fromkeys deduplica
    # DESPUÉS del recorte y deja las líneas en orden creciente.
    for n in dict.fromkeys(sorted(min(n, len(conteo)) for n in (10, top))):
        print(f"Cobertura del top-{n}: {resumen['pct_acum'].iloc[n - 1]}%")

    return resumen


def graficar_top(conteo, titulo, top=20):
    """Dot plot del top-N de clases, en escala logarítmica.

    Punto y no barra a propósito: el rango de estas variables abarca varios órdenes de
    magnitud y necesita eje logarítmico, pero una barra codifica LONGITUD DESDE CERO y la
    escala log no tiene cero — la barra de una clase con 171 casos se vería como un quinto
    de otra con 33.268, no como un 195-avo. El punto codifica posición, no longitud, así
    que sobre un eje log sigue diciendo la verdad.
    """
    datos = conteo.head(top).sort_values()
    posiciones = range(len(datos))

    fig, ax = plt.subplots(figsize=(9, max(4, len(datos) * 0.32)))

    # Línea guía tenue desde el borde del gráfico hasta el punto: ayuda a seguir la fila
    # sin competir visualmente con el dato.
    ax.hlines(posiciones, datos.min() * 0.85, datos.values, color="#D0D0D0", linewidth=1)
    ax.plot(datos.values, posiciones, "o", markersize=8, color=COLOR_DATO)

    ax.set_yticks(list(posiciones), datos.index.astype(str))
    ax.set_xscale("log")
    ax.set_xlabel("Cantidad de registros (escala log)")
    ax.set_title(titulo)

    # Etiqueta directa al lado de cada punto: evita leer el eje para cada valor
    for y, valor in zip(posiciones, datos.values):
        ax.text(valor * 1.12, y, f"{valor:,}", va="center", fontsize=8, color="#444444")

    # Ejes recesivos: el dato es el punto, no el marco
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    # Grilla vertical suave por detrás, como referencia de magnitud
    ax.grid(axis="x", color="#EEEEEE", linewidth=1)
    ax.set_axisbelow(True)
    # Margen a ambos lados para que ni la guía ni la etiqueta directa se corten
    ax.set_xlim(datos.min() * 0.8, datos.max() * 2.2)

    plt.tight_layout()
    plt.show()

###### 5.1 `materia`

Es texto libre cargado por el analista, no un catálogo cerrado: aparecen combinaciones
(`CIVIL - COMERCIAL`, `PENAL-PROCESAL`) y el mismo valor escrito de varias formas
(`CIVIL-COMERCIAL` vs `CIVIL - COMERCIAL`). Contamos primero el valor crudo y después una
versión normalizada, para ver cuánto del "desbalance" es en realidad suciedad de carga.

In [ ]:
# Valor crudo, tal como viene en el JSON. dropna() saca los 391.540 registros sin materia
# (son documentos que no son sumarios: la variable solo existe para una parte del dataset).
materia_cruda = df_clases["materia"].dropna()

resumen_materia_cruda = resumen_desbalance(materia_cruda, "materia (valor crudo)")

with pd.option_context("display.max_rows", 40):
    display(resumen_materia_cruda.head(40))

In [ ]:
import re


def normalizar_materia(valor):
    """Parte una materia compuesta en sus materias simples y las estandariza.

    Separamos por guion, coma y barra, pero NO por espacio: hay materias legítimas de
    dos palabras (`SEGURIDAD SOCIAL`, `DERECHOS HUMANOS`) que se romperían.
    """
    # upper() unifica mayúsculas/minúsculas antes de comparar
    partes = re.split(r"[-,/]", valor.upper())
    # \s+ colapsa espacios repetidos: "CIVIL   COMERCIAL" y "CIVIL COMERCIAL" son lo mismo
    limpias = {re.sub(r"\s+", " ", p).strip() for p in partes}
    # Descartamos cadenas vacías que deja el split cuando hay separadores seguidos
    return sorted(p for p in limpias if p)


# Una lista de materias por registro; explode() genera una fila por materia.
# Con esto pasamos de "multiclase con clases compuestas" a "multietiqueta de materias simples".
materias_normalizadas = materia_cruda.map(normalizar_materia).explode()

resumen_materia = resumen_desbalance(materias_normalizadas, "materia (normalizada)")

with pd.option_context("display.max_rows", 40):
    display(resumen_materia.head(40))

graficar_top(resumen_materia["n"], "Top 20 materias (normalizadas)")

###### 5.2 `descriptores`

La estructura anidada es:

```
{"descriptor": [{"elegido":   {"termino": "despido"},
                 "preferido": {"termino": "Derecho laboral/contrato de trabajo/.../despido"},
                 "sinonimos": {"termino": [...]}}, ...],
 "suggest": {"termino": [...]}}
```

Nos quedamos con `elegido.termino` (el término del tesauro asignado al sumario). Es **multietiqueta**:
un sumario tiene varios descriptores, así que la suma de porcentajes no es sobre registros sino sobre
etiquetas. Además guardamos `preferido.termino`, cuyo primer tramo antes de la `/` es la rama del
tesauro: una jerarquía ya construida que sirve como etiqueta más gruesa y mucho menos desbalanceada.

Ojo con un valor centinela: hay registros plantilla con el literal
`"Término elegido para describir al caso"`. No es un descriptor, hay que filtrarlo.

In [ ]:
# Textos plantilla que el dataset trae en registros de ejemplo: no son descriptores reales
CENTINELAS_DESCRIPTOR = {
    "Término elegido para describir al caso",
    "Término preferido para describir al caso",
}


def extraer_terminos(valor, clave):
    """Devuelve la lista de términos bajo `clave` ('elegido' o 'preferido') de un registro."""
    if not isinstance(valor, dict):
        return []

    descriptores = valor.get("descriptor")
    # Cuando hay un solo descriptor el JSON trae un dict en lugar de una lista de un elemento
    if isinstance(descriptores, dict):
        descriptores = [descriptores]
    if not isinstance(descriptores, list):
        return []

    terminos = []
    for d in descriptores:
        if not isinstance(d, dict):
            continue
        # Doble get(): 'elegido' puede faltar, y dentro puede faltar 'termino'
        termino = (d.get(clave) or {}).get("termino")
        # Descartamos vacíos y plantillas en el mismo paso
        if isinstance(termino, str) and termino.strip() and termino not in CENTINELAS_DESCRIPTOR:
            terminos.append(termino.strip())

    # set() evita contar dos veces el mismo término repetido dentro de un registro
    return sorted(set(terminos))


descriptores_validos = df_clases["descriptores"].dropna()

# Desanidamos UNA sola vez y guardamos las listas: recorrer la estructura anidada es lo caro
# de esta sección, y de acá salen tanto el conteo por registro como el explode de más abajo.
elegidos_por_registro = descriptores_validos.map(lambda v: extraer_terminos(v, "elegido"))

# Cuántas etiquetas tiene cada registro: define si el problema es multietiqueta y de qué tamaño
terminos_por_registro = elegidos_por_registro.map(len)
print(f"Registros con descriptores: {len(descriptores_validos):,}")
print(f"Registros que quedan en 0 tras filtrar plantillas: {(terminos_por_registro == 0).sum():,}")
print(f"Descriptores por registro — mediana: {terminos_por_registro.median():.0f} | máximo: {terminos_por_registro.max()}")

In [ ]:
# Reutilizamos las listas ya desanidadas en la celda anterior; explode() deja una fila por término
terminos_elegidos = elegidos_por_registro.explode().dropna()

resumen_descriptores = resumen_desbalance(terminos_elegidos, "descriptores — término elegido")

with pd.option_context("display.max_rows", 40):
    display(resumen_descriptores.head(40))

graficar_top(resumen_descriptores["n"], "Top 20 descriptores (término elegido)")

In [ ]:
# Rama del tesauro: el primer tramo de "Derecho laboral/contrato de trabajo/.../despido".
# Es la misma información agrupada a un nivel más grueso, con muchas menos clases.
ramas_tesauro = (
    descriptores_validos.map(lambda v: sorted({t.split("/")[0].strip() for t in extraer_terminos(v, "preferido")}))
    .explode()
    .dropna()
)

resumen_ramas = resumen_desbalance(ramas_tesauro, "descriptores — rama del tesauro")

with pd.option_context("display.max_rows", None):
    display(resumen_ramas)

graficar_top(resumen_ramas["n"], "Ramas del tesauro (término preferido)")

###### 5.3 `tipo-tribunal`

Códigos cortos (`CS` = Corte Suprema, `CC` = Civil y Comercial, `LB` = Laboral, ...). El campo debería
ser un catálogo cerrado, pero trae ruido de carga: códigos repetidos en la misma celda (`CS CS`),
espacios de más y variantes de mayúsculas. Normalizamos antes de contar.

In [ ]:
def normalizar_tipo_tribunal(valor):
    """Deduplica códigos repetidos dentro de la misma celda y unifica espacios."""
    # dict.fromkeys() elimina duplicados CONSERVANDO el orden: "CS CS" -> "CS",
    # "CC CO" -> "CC CO" (una combinación real de fueros, no un duplicado).
    codigos = dict.fromkeys(valor.upper().split())
    return " ".join(codigos)


tipo_tribunal_crudo = df_clases["tipo-tribunal"].dropna()

print(f"Valores distintos sin normalizar: {tipo_tribunal_crudo.nunique():,}")

tipo_tribunal = tipo_tribunal_crudo.map(normalizar_tipo_tribunal)

resumen_tribunal = resumen_desbalance(tipo_tribunal, "tipo-tribunal")

with pd.option_context("display.max_rows", 40):
    display(resumen_tribunal.head(40))

graficar_top(resumen_tribunal["n"], "Top 20 tipos de tribunal")

###### 5.4 `jurisdiccion`

Viene como diccionario: `{"codigo": "NACIONAL", "descripcion": "Nacional", "capital": "Nacional", "id-pais": 11}`.
Usamos `descripcion`. Es la variable de menor cardinalidad de las cuatro, así que la mostramos completa
y en escala lineal: acá el gráfico sí puede mostrar la proporción real sin aplastar nada.

In [ ]:
def extraer_jurisdiccion(valor):
    """Devuelve la descripción legible de la jurisdicción."""
    if isinstance(valor, dict):
        # Si faltara 'descripcion' caemos a 'codigo', que es el mismo dato en formato crudo
        return valor.get("descripcion") or valor.get("codigo")
    # Algunos registros podrían traer el string directo en vez del dict
    return valor if isinstance(valor, str) else None


jurisdiccion = df_clases["jurisdiccion"].map(extraer_jurisdiccion).dropna()

resumen_jurisdiccion = resumen_desbalance(jurisdiccion, "jurisdiccion", top=5)

display(resumen_jurisdiccion)

# Escala lineal y orden descendente: con pocas clases la comparación de magnitudes es directa
datos = resumen_jurisdiccion["n"].sort_values()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(datos.index, datos.values, color=COLOR_DATO)
ax.set_xlabel("Cantidad de registros")
ax.set_title("Jurisdicción")
for y, valor in enumerate(datos.values):
    ax.text(valor * 1.02, y, f"{valor:,}", va="center", fontsize=8, color="#444444")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.set_xlim(right=datos.max() * 1.15)
plt.tight_layout()
plt.show()

###### 5.5 Comparación entre las cuatro variables

Puestas una al lado de la otra se ve el compromiso: cuanto más fina la etiqueta, más informativa para
el usuario final y más desbalanceada para el modelo.

In [ ]:
candidatas = {
    "materia (cruda)": resumen_materia_cruda,
    "materia (normalizada)": resumen_materia,
    "descriptores (término)": resumen_descriptores,
    "descriptores (rama tesauro)": resumen_ramas,
    "tipo-tribunal": resumen_tribunal,
    "jurisdiccion": resumen_jurisdiccion,
}


def clases_para_cobertura(resumen, umbral=90):
    """Cuántas clases hacen falta para cubrir `umbral`% de las etiquetas."""
    # Contamos las que quedan por debajo del umbral y sumamos 1: esa es la que lo cruza.
    # min() evita devolver len+1 cuando ninguna clase llega al umbral (colas muy largas).
    return min(int((resumen["pct_acum"] < umbral).sum() + 1), len(resumen))


comparacion = pd.DataFrame(
    [
        {
            "variable": nombre,
            "clases": len(r),
            # En las variables multietiqueta este total supera la cantidad de registros:
            # cada registro aporta varias etiquetas.
            "etiquetas_totales": int(r["n"].sum()),
            "pct_clase_mayoritaria": r["pct"].iloc[0],
            "ratio_max_min": round(r["n"].max() / r["n"].min()),
            "clases_con_menos_de_50": int((r["n"] < MIN_EJEMPLOS_CLASE).sum()),
            "clases_para_90pct": clases_para_cobertura(r),
        }
        for nombre, r in candidatas.items()
    ]
)

display(comparacion)